# 🚀 Deriv Hybrid Predictor - Colab API

**Run the entire prediction engine on Google Colab's free GPU**

---

## 📋 Before You Start

1. **Enable GPU**: Runtime → Change runtime type → GPU
2. **Get ngrok token**: https://dashboard.ngrok.com/get-started/your-authtoken
3. **Upload project to Drive**: Upload the `der` folder to Google Drive

---

## ▶️ Run All Cells Below

## 1️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

## 2️⃣ Locate Project

**IMPORTANT**: Adjust the path below to match where you uploaded the project in Google Drive

In [ ]:
import os
import sys

# Try common locations
possible_paths = [
    '/content/drive/MyDrive/der',
    '/content/drive/MyDrive/Deriv_Predictor',
    '/content/drive/MyDrive/2026/der',
    '/content/drive/My Drive/der',
    '/content/drive/My Drive/Deriv_Predictor'
]

PROJECT_PATH = None

# Find project
for path in possible_paths:
    if os.path.exists(path):
        PROJECT_PATH = path
        break

if PROJECT_PATH:
    print(f"✅ Project found at: {PROJECT_PATH}")
    os.chdir(PROJECT_PATH)
    print(f"📁 Working directory: {os.getcwd()}")
    
    # List contents to verify
    print("\n📂 Project contents:")
    for item in os.listdir('.'):
        print(f"   - {item}")
else:
    print("❌ Project not found!")
    print("\n📍 Please upload the 'der' folder to one of these locations:")
    for path in possible_paths:
        print(f"   - {path}")
    print("\n💡 Or manually set PROJECT_PATH below:")
    print("   PROJECT_PATH = '/content/drive/MyDrive/YOUR_FOLDER_NAME'")
    
    # Let user set custom path
    custom_path = input("\nEnter custom path (or press Enter to skip): ").strip()
    if custom_path and os.path.exists(custom_path):
        PROJECT_PATH = custom_path
        os.chdir(PROJECT_PATH)
        print(f"✅ Using custom path: {PROJECT_PATH}")
    else:
        raise Exception("Project path not found. Please upload the project to Google Drive first.")

## 3️⃣ Install Dependencies

In [ ]:
print("📦 Installing dependencies...")

# Install from requirements.txt if it exists
if os.path.exists('requirements.txt'):
    !pip install -q -r requirements.txt
else:
    # Install manually
    !pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

# Install API dependencies
!pip install -q fastapi uvicorn pyngrok nest-asyncio

print("✅ Dependencies installed")

## 4️⃣ Check GPU Availability

In [ ]:
import tensorflow as tf

# Check GPU
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"✅ GPU available: {gpus[0].name}")
    print(f"   Type: {gpus[0].device_type}")
else:
    print("⚠️ No GPU found!")
    print("   Go to: Runtime → Change runtime type → GPU")

# Show TensorFlow version
print(f"\n📦 TensorFlow version: {tf.__version__}")

## 5️⃣ Load Core Engine

In [ ]:
import sys
import os

# Add project to Python path
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print(f"📍 Python path: {PROJECT_PATH}")

# Verify core folder exists
core_path = os.path.join(PROJECT_PATH, 'core')
if not os.path.exists(core_path):
    raise Exception(f"'core' folder not found at: {core_path}")

print(f"✅ Core folder found: {core_path}")

# Import core engine
try:
    from core import HybridEngine
    from core.config import DERIV_CONFIG, BUFFER_SIZE, RETRAIN_CONFIG
    
    print("✅ Core engine imported successfully")
    print(f"   Buffer size: {BUFFER_SIZE}")
    print(f"   Retrain interval: {RETRAIN_CONFIG['tick_interval']}")
    print(f"   Symbol: {DERIV_CONFIG['symbol']}")
    
except ImportError as e:
    print(f"❌ Failed to import core engine: {e}")
    print(f"\n🔍 Debugging info:")
    print(f"   Current directory: {os.getcwd()}")
    print(f"   Python path: {sys.path[:3]}")
    print(f"   Core exists: {os.path.exists('core')}")
    if os.path.exists('core'):
        print(f"   Core contents: {os.listdir('core')[:5]}")
    raise

## 6️⃣ Initialize Engine

In [ ]:
from datetime import datetime

# Create model save directory
MODEL_DIR = os.path.join(PROJECT_PATH, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print("🧠 Initializing Hybrid Engine...")

# Initialize engine
engine = HybridEngine(verbose=True)

print(f"✅ Engine initialized")
print(f"   Model save path: {MODEL_DIR}")
print(f"   Timestamp: {datetime.now()}")

# Try to load existing models
try:
    lstm_path = os.path.join(MODEL_DIR, 'lstm_model.h5')
    if os.path.exists(lstm_path):
        engine.lstm_engine.load_model(lstm_path)
        print("📦 Loaded existing LSTM model")
        engine.is_trained = True
except Exception as e:
    print(f"⚠️ No existing models found (will train from scratch)")

## 7️⃣ Create FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Optional
import nest_asyncio

# Allow nested event loops (required for Colab)
nest_asyncio.apply()

# Create FastAPI app
app = FastAPI(
    title="Deriv Hybrid Predictor API",
    description="GPU-accelerated prediction engine running on Google Colab",
    version="1.0.0"
)

# Request/Response models
class Tick(BaseModel):
    timestamp: int
    quote: float
    symbol: str

class PredictRequest(BaseModel):
    ticks: List[Tick]

class PredictResponse(BaseModel):
    prediction: Optional[Dict]
    status: Dict
    timestamp: str

# Endpoints
@app.get("/")
async def root():
    return {
        "name": "Deriv Hybrid Predictor",
        "status": "running",
        "gpu_available": len(tf.config.list_physical_devices('GPU')) > 0,
        "engine_trained": engine.is_trained,
        "tick_count": engine.tick_count
    }

@app.post("/predict", response_model=PredictResponse)
async def predict(request: PredictRequest):
    """
    Get prediction for given ticks
    """
    try:
        # Add ticks to engine
        for tick in request.ticks:
            engine.add_tick(tick.dict())
        
        # Train if needed
        if not engine.is_trained and engine.tick_count >= 500:
            print("🚀 Starting initial training...")
            results = engine.initial_train()
            print(f"✅ Training complete: {results['lstm_accuracy']:.2%}")
            
            # Save models
            engine.lstm_engine.save_model(os.path.join(MODEL_DIR, 'lstm_model.h5'))
            print("💾 Models saved")
        
        # Get prediction
        prediction = None
        if engine.is_trained:
            prediction = engine.predict_next_tick()
        
        return PredictResponse(
            prediction=prediction,
            status={
                "is_trained": engine.is_trained,
                "tick_count": engine.tick_count,
                "gpu_available": len(tf.config.list_physical_devices('GPU')) > 0
            },
            timestamp=datetime.now().isoformat()
        )
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/status")
async def get_status():
    """
    Get engine status
    """
    return engine.get_status()

@app.post("/retrain")
async def force_retrain():
    """
    Force model retraining
    """
    if not engine.is_trained:
        raise HTTPException(400, "Engine not trained yet")
    
    print("🔄 Forcing retrain...")
    results = engine.retrain()
    
    # Save models
    engine.lstm_engine.save_model(os.path.join(MODEL_DIR, 'lstm_model.h5'))
    
    return {
        "message": "Retrain complete",
        "results": results
    }

print("✅ FastAPI server created")
print("   Endpoints: /, /predict, /status, /retrain")

## 8️⃣ Setup ngrok (Public URL)

In [ ]:
from pyngrok import ngrok
import getpass

# Get ngrok token
print("🔑 Get your ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("Enter your ngrok authtoken: ")

# Set token
ngrok.set_auth_token(ngrok_token)

print("✅ ngrok token set")

## 9️⃣ Start Server

In [ ]:
import uvicorn
from threading import Thread
import time

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

# Create ngrok tunnel
public_url = ngrok.connect(8000)

print("=" * 70)
print("🎉 SERVER RUNNING!")
print("=" * 70)
print(f"\n📡 Public URL: {public_url}")
print(f"\n🔗 COPY THIS URL and use it in your local backend!")
print(f"\n📝 In backend/main.py, set:")
print(f"   COLAB_URL = \"{public_url}\"")
print(f"\n📚 API Docs: {public_url}/docs")
print(f"\n✅ Test endpoint: {public_url}/")
print("\n" + "=" * 70)
print("\n⚠️ Keep this notebook running!")
print("   The server will stop if you close this tab.")
print("\n💡 Tip: Use Colab Pro for 24-hour sessions")
print("=" * 70)

## 🔟 Monitor Server (Optional)

In [ ]:
import time
from IPython.display import clear_output

# Monitor loop
try:
    while True:
        clear_output(wait=True)
        
        status = engine.get_status()
        
        print("=" * 70)
        print("📊 DERIV PREDICTOR - LIVE STATUS")
        print("=" * 70)
        print(f"\n🔗 Public URL: {public_url}")
        print(f"\n🧠 Engine Status:")
        print(f"   Trained: {status['is_trained']}")
        print(f"   Tick count: {status['tick_count']}")
        print(f"   Predictions made: {status.get('prediction_count', 0)}")
        
        if status['is_trained']:
            print(f"\n📈 Performance:")
            print(f"   Accuracy (50): {status.get('accuracy_50', 0):.2%}")
            print(f"   Accuracy (200): {status.get('accuracy_200', 0):.2%}")
        
        print(f"\n⏰ Last update: {datetime.now().strftime('%H:%M:%S')}")
        print("\n💡 Press Ctrl+C to stop monitoring (server keeps running)")
        print("=" * 70)
        
        time.sleep(5)

except KeyboardInterrupt:
    print("\n✅ Monitoring stopped (server still running)")

## 1️⃣1️⃣ Save Models (Manual)

In [ ]:
# Manually save models to Google Drive
if engine.is_trained:
    print("💾 Saving models...")
    
    # Save LSTM
    lstm_path = os.path.join(MODEL_DIR, 'lstm_model.h5')
    engine.lstm_engine.save_model(lstm_path)
    print(f"   ✅ LSTM saved: {lstm_path}")
    
    # Save metadata
    import json
    metadata = {
        "tick_count": engine.tick_count,
        "is_trained": engine.is_trained,
        "timestamp": datetime.now().isoformat()
    }
    
    with open(os.path.join(MODEL_DIR, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print("   ✅ Metadata saved")
    print("\n✅ All models saved to Google Drive!")
else:
    print("⚠️ Engine not trained yet")

---

## 🎉 You're Done!

Your prediction engine is now running on Google Colab's free GPU!

**Next steps**:
1. Copy the ngrok URL from above
2. Update your local backend (`backend/main.py`) with the URL
3. Start your local backend
4. Start receiving predictions!

**Keep this notebook running** to keep the server alive.